# Sanity 5: Neighborhood Overlap
Compares k-NN overlap under identity vs random projection/re-embedding.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)
N, d, m, k = 500, 64, 16, 10
X = torch.randn(N, d)

def knn_idx(P, k):
    D = torch.cdist(P, P)
    D.fill_diagonal_(float('inf'))
    return torch.topk(D, k=k, largest=False).indices

def overlap(A, B, k):
    IA, IB = knn_idx(A, k), knn_idx(B, k)
    vals = []
    for i in range(A.shape[0]):
        sa, sb = set(IA[i].tolist()), set(IB[i].tolist())
        vals.append(len(sa & sb) / k)
    return float(torch.tensor(vals).mean())

Q, _ = torch.linalg.qr(torch.randn(d, m))
R = torch.randn(m, d)
Y_identity = X.clone()
Y_proj = (X @ Q) @ R

ov_id = overlap(X, Y_identity, k)
ov_proj = overlap(X, Y_proj, k)
print(f'Identity overlap: {ov_id:.3f}')
print(f'Projection overlap: {ov_proj:.3f}')

plt.figure(figsize=(5, 4))
plt.bar(['identity', 'projection'], [ov_id, ov_proj], color=['#2ca02c', '#d62728'])
plt.ylim(0, 1.05)
plt.ylabel('Mean k-NN overlap')
plt.title('Neighborhood overlap sanity check')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()